# Retail Data Exploration

# TODO change all df to df_raw

## Introduction

This notebook explores the 2009–2010 portion of the UCI Online Retail II dataset using Python and Pandas.

It includes data completeness and quality assessments and informs the design of the reusable data cleaning pipeline.

## Load the data

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [20]:
df = pd.read_excel("../data/raw/online_retail/online_retail_II.xlsx")

## Dataset overview

In [21]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Each row represents a product purchased within an invoice.

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 32.1+ MB


In [23]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845018,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


The project uses the 2009–2010 sheet from the UCI Online Retail II dataset. It contains transaction-level retail data from a UK online retailer recorded between 1 December 2009 and 9 December 2010.

It contains both numerical and categorical variables describing invoices, products, customers and transactions. Summary statistics show both negative and unusually large positive quantities and prices, which are investigated in the following sections.

## Data quality

### Data completeness

The dataset overview showed that the latest transaction was recorded on 9 December 2010. The number of transactions per month was examined to understand the coverage of the selected data.

In [24]:
df.groupby(df["InvoiceDate"].dt.to_period("M")).size()

InvoiceDate
2009-12    45228
2010-01    31555
2010-02    29388
2010-03    41511
2010-04    34057
2010-05    35323
2010-06    39983
2010-07    33383
2010-08    33306
2010-09    42091
2010-10    59098
2010-11    78015
2010-12    22523
Freq: M, dtype: int64

December 2010 contains far fewer transactions than the preceding months, confirming that the selected data includes only the first nine days of December. Due to this, December 2010 will be excluded from monthly comparison analysis.

### Missing values

In [25]:
# check number of missing values
df.isna().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [26]:
#  check percentage of missing values
(df.isna().sum() / len(df) * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.56
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    20.54
Country         0.00
dtype: float64

Customer ID contains a substantial number of missing values, around 21%. Description has relatively few missing values only around 0.6%. Other columns appear complete.

### Data types

In [27]:
# check data types
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

All of the data types are correct, no conversions needed.

### Duplicates

In [28]:
df.duplicated().sum()

np.int64(6865)

The dataset contains 6,865 duplicate rows. These will be removed during the data cleaning stage to avoid double-counting transactions.

### Investigate unusual quantities

#### Large quantities

In [29]:
df["Quantity"].describe()

count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

In [30]:
# view the largest quantities
df.nlargest(20,"Quantity")[["Invoice", "Customer ID", "StockCode", "Description", "Quantity", "Price", "Country"]]

,Invoice,Customer ID,StockCode,Description,Quantity,Price,Country
90857,497946,13902.0,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,Denmark
127166,501534,13902.0,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,Denmark
127168,501534,13902.0,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,Denmark
127169,501534,13902.0,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,Denmark
127167,501534,13902.0,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,Denmark
192197,507637,NaN,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,United Kingdom
135027,502269,17940.0,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,United Kingdom
135028,502269,17940.0,21982,PACK OF 12 SUKI TISSUES,10000,0.25,United Kingdom
135029,502269,17940.0,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,United Kingdom
135030,502269,17940.0,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,United Kingdom


Very large order quantities appear to represent genuine bulk purchases of low cost products, rather than obvious data errors, so these will be retained.

#### Negative quantities

In [31]:
# number of negative quantitiea
(df["Quantity"] < 0).sum()

np.int64(12326)

In [32]:
df[df["Quantity"] < 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


There are 12,326 rows with negative quantities, which appear to represent cancelled orders or product returns, as the sampled records have invoice numbers beginning with "C".

### Investigate unusual prices

#### Large prices

In [33]:
df["Price"].describe()

count    525461.000000
mean          4.688834
std         146.126914
min      -53594.360000
25%           1.250000
50%           2.100000
75%           4.210000
max       25111.090000
Name: Price, dtype: float64

In [34]:
df.nlargest(20,"Price")[["Invoice", "StockCode", "Description", "Quantity", "Price", "Country"]]

,Invoice,StockCode,Description,Quantity,Price,Country
241824,C512770,M,Manual,-1,25111.09,United Kingdom
241827,512771,M,Manual,1,25111.09,United Kingdom
320581,C520667,BANK CHARGES,Bank Charges,-1,18910.69,United Kingdom
517953,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33,United Kingdom
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,United Kingdom
519294,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33,United Kingdom
519170,C537644,AMAZONFEE,AMAZON FEE,-1,13474.79,United Kingdom
135012,C502262,M,Manual,-1,10953.50,United Kingdom
135013,502263,M,Manual,1,10953.50,United Kingdom
135014,C502264,M,Manual,-1,10953.50,United Kingdom


A number of administrative records (e.g. Manual, Adjustment and Amazon Fee entries) with unusually high prices, and some with negative quantities, were identified. These will be retained for this version of the project because they represent valid records rather than errors.

#### Negative prices

In [35]:
(df["Price"] < 0).sum()

np.int64(3)

In [36]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


Only three records contain negative prices. All are labelled "Adjust bad debt", suggesting these are accounting adjustments rather than product sales. These records will be removed during data cleaning.

## Key raw data findings

- The dataset contained complete transaction data from December 2009 to November 2010. December 2010 contains only the first nine days of transactions.
- The dataset contained:
  - Duplicate records. 
  - Cancelled or returned transactions.
  - Three bad debt adjustment transactions with negative prices.
  - Missing product descriptions and customer identifiers.

These findings informed the design of the reusable data cleaning pipeline implemented in ```clean_data.py.```